# BRKGA Foco — Flow Shop Scheduling (NPFS)

Comparação das 3 variantes do BRKGA (fixo_default, fixo_optuna, fuzzy)
em **todas as instâncias 16×** (16 jobs, 2–5 máquinas, grupos 2–4).

**Fluxo:**
1. Tuning com Optuna na instância alvo 16×5
2. Calibração das referências fuzzy
3. Loop comparativo: 60 instâncias × 5 seeds × 3 variantes
4. Resultado consolidado em CSV com checkpoint que nao sei se funciona

## 0. Setup — Instalação e Imports

In [1]:
#instala dep.
!pip install -q pymoo scikit-fuzzy optuna numpy pandas
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencias instaladas.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.1 MB/s eta 0:00:00
Dependencias instaladas.


In [2]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, time, json, csv
sys.path.insert(0, ROOT)

from pathlib import Path
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import optuna

DATA_DIR = Path(ROOT) / 'data' / 'Small'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_foco'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Resultados em: {RESULTADOS_DIR}')

ROOT: /content/drive/MyDrive/FlowShopScheduling
DATA_DIR: /content/drive/MyDrive/FlowShopScheduling/data/Small
Resultados em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_foco


In [4]:
# Imports
from src.flowshop_engine import FlowShopEngine
from src.solvers.adapter import rodar_brkga, SolverResult
from src.solvers.brkga_solver import (
    rodar_optuna,
    analisar_melhor_configuracao,
    construir_referencia_a_partir_da_trajetoria,
)

print('Imports concluídos.')

Imports concluídos.


## 1. Catálogo de Instâncias — 60 instâncias 16×

Extrai todas as instâncias com 16 jobs do diretório `data/Small/`.
Parseia o nome do arquivo: `I_{grupo}_{n_jobs}_{n_machines}_{id}.txt`.

In [5]:
def parse_nome_instancia(nome: str) -> Dict:
    """
    Converte 'I_2_16_5_1.txt' -> {grupo: 2, n_jobs: 16, n_machines: 5, id: 1}
    """
    stem = nome.replace('.txt', '')
    partes = stem.split('_')
    return {
        'grupo': int(partes[1]),
        'n_jobs': int(partes[2]),
        'n_machines': int(partes[3]),
        'id': int(partes[4]),
        'arquivo': nome,
    }


def construir_catalogo_16x(data_dir: Path) -> pd.DataFrame:
    """
    Retorna DataFrame com todas as instancias de 16 jobs.
    """
    todos = sorted(data_dir.glob('I_*_16_*.txt'))
    registros = []
    for f in todos:
        info = parse_nome_instancia(f.name)
        info['path'] = str(f)
        registros.append(info)

    df = pd.DataFrame(registros)
    df = df.sort_values(['grupo', 'n_machines', 'id']).reset_index(drop=True)
    df['rotulo'] = df.apply(
        lambda r: f'{r["n_jobs"]}x{r["n_machines"]}_g{r["grupo"]}_id{r["id"]}', axis=1
    )
    return df


catalogo = construir_catalogo_16x(DATA_DIR)
print(f'Total de instancias 16x: {len(catalogo)}')
print()
print('Distribuicao:')
for m in sorted(catalogo['n_machines'].unique()):
    qtd = len(catalogo[catalogo['n_machines'] == m])
    print(f'  16x{m}: {qtd} instancias')
print()
print('10 primeiras:')
print(catalogo.head(10).to_string(index=False))

Total de instancias 16x: 60

Distribuicao:
  16x2: 15 instancias
  16x3: 15 instancias
  16x4: 15 instancias
  16x5: 15 instancias

10 primeiras:
 grupo  n_jobs  n_machines  id        arquivo                                                                path      rotulo
     2      16           2   1 I_2_16_2_1.txt /content/drive/MyDrive/FlowShopScheduling/data/Small/I_2_16_2_1.txt 16x2_g2_id1
     2      16           2   2 I_2_16_2_2.txt /content/drive/MyDrive/FlowShopScheduling/data/Small/I_2_16_2_2.txt 16x2_g2_id2
     2      16           2   3 I_2_16_2_3.txt /content/drive/MyDrive/FlowShopScheduling/data/Small/I_2_16_2_3.txt 16x2_g2_id3
     2      16           2   4 I_2_16_2_4.txt /content/drive/MyDrive/FlowShopScheduling/data/Small/I_2_16_2_4.txt 16x2_g2_id4
     2      16           2   5 I_2_16_2_5.txt /content/drive/MyDrive/FlowShopScheduling/data/Small/I_2_16_2_5.txt 16x2_g2_id5
     2      16           3   1 I_2_16_3_1.txt /content/drive/MyDrive/FlowShopScheduling/data/Small

## 2. Configuração Global

In [6]:
# Parametros do BRKGA
POP_SIZE = 100
N_ELITES_DEFAULT = 20
N_OFFSPRINGS_DEFAULT = 70
N_MUTANTS_DEFAULT = 10
BIAS_DEFAULT = 0.7

# Geracoes
N_GEN = 100  # 100 gen x 100 pop = 10.000 avaliacoes

N_SEEDS = 5
SEED_INICIO = 100

# Objetivo
OBJECTIVE = 'Makespan'

# Tuning
INSTANCIA_ALVO = 'I_2_16_5_1.txt'   # alvo principal (v5)
N_TRIALS_OPTUNA = 15
SEEDS_TUNING = (1, 2, 3)            # 3 seeds por trial

print('Configuracao:')
print(f'  Populacao: {POP_SIZE} ({N_ELITES_DEFAULT} elites, {N_OFFSPRINGS_DEFAULT} offsprings, {N_MUTANTS_DEFAULT} mutantes)')
print(f'  Geracoes: {N_GEN}')
print(f'  Avaliacoes/exec: {N_GEN * POP_SIZE}')
print(f'  Seeds: {N_SEEDS}  ({SEED_INICIO}..{SEED_INICIO + N_SEEDS - 1})')
print(f'  Objetivo: {OBJECTIVE}')
print(f'  Instancia alvo tuning: {INSTANCIA_ALVO}')
print(f'  Trials Optuna: {N_TRIALS_OPTUNA}')

Configuracao:
  Populacao: 100 (20 elites, 70 offsprings, 10 mutantes)
  Geracoes: 100
  Avaliacoes/exec: 10000
  Seeds: 5  (100..104)
  Objetivo: Makespan
  Instancia alvo tuning: I_2_16_5_1.txt
  Trials Optuna: 15


## 3. Tuning com Optuna — Instância Alvo 16×5

Roda o Optuna na instância alvo (`I_2_16_5_1.txt`)
para encontrar os melhores parâmetros: `frac_elites`, `frac_mutantes`, `bias`.

O resultado é salvo em JSON para reutilizacao

In [7]:
# Carrega a engine da instancia alvo
engine_alvo = FlowShopEngine(
    *FlowShopEngine.carregar_instancia_txt(
        str(DATA_DIR / INSTANCIA_ALVO)
    )
)
print(f'Engine alvo: {engine_alvo.n_jobs} jobs, {engine_alvo.n_machines} maquinas')

Engine alvo: 16 jobs, 5 maquinas


In [8]:
print('>>> Iniciando tuning com Optuna...')
t0_tuning = time.time()

study, melhor_config = rodar_optuna(
    engine_alvo,
    pop_size=POP_SIZE,
    n_gen=N_GEN,           # 100 gen durante o tuning tambem
    seeds=SEEDS_TUNING,
    n_trials=N_TRIALS_OPTUNA,
    seed_optuna=0,
)

elapsed_tuning = time.time() - t0_tuning
print(f'Tuning concluido em {elapsed_tuning:.1f}s')
print(f'Melhor custo: {study.best_value:.2f}')
print(f'Melhor config: {melhor_config}')

# Salva config em JSON
config_path = RESULTADOS_DIR / 'config_optuna.json'
with open(config_path, 'w') as f:
    json.dump(melhor_config, f, indent=2)
print(f'Config salva em: {config_path}')

[I 2026-09-29 01:08:17,730] A new study created in memory with name: no-name-8a77ed63-1b87-4872-ade1-34c44cb6b507


>>> Iniciando tuning com Optuna...


[I 2026-09-29 01:08:25,426] Trial 0 finished with value: 1546.0 and parameters: {'frac_elites': 0.24208472637456369, 'frac_mutantes': 0.30031627823034684, 'bias': 0.6712435192322397}. Best is trial 0 with value: 1546.0.
[I 2026-09-29 01:08:33,160] Trial 1 finished with value: 1446.6666666666667 and parameters: {'frac_elites': 0.24070911404891393, 'frac_mutantes': 0.19827917976861664, 'bias': 0.6906523508799953}. Best is trial 1 with value: 1446.6666666666667.
[I 2026-09-29 01:08:40,780] Trial 2 finished with value: 1536.3333333333333 and parameters: {'frac_elites': 0.20315552394194242, 'frac_mutantes': 0.3621205502737279, 'bias': 0.8336482422254632}. Best is trial 1 with value: 1446.6666666666667.
[I 2026-09-29 01:08:49,485] Trial 3 finished with value: 1557.6666666666667 and parameters: {'frac_elites': 0.18420453158902222, 'frac_mutantes': 0.3271037633289326, 'bias': 0.638002713888807}. Best is trial 1 with value: 1446.6666666666667.
[I 2026-09-29 01:08:56,025] Trial 4 finished with v

Tuning concluido em 122.1s
Melhor custo: 1354.33
Melhor config: {'n_elites': 8, 'n_offsprings': 86, 'n_mutants': 6, 'bias': 0.7746789304965721}
Config salva em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_foco/config_optuna.json


## 4. Calibração das Referências Fuzzy

Usa a melhor config do Optuna para gerar as trajetórias medias
de diversidade e qualidade. Essas trajetórias servem como
referencia para os controladores fuzzy

In [9]:
print('>>> Calibrando referencias fuzzy...')
t0_ref = time.time()

_, traj_div, traj_qual, resumo_calib = analisar_melhor_configuracao(
    engine_alvo,
    melhor_config,
    n_gen=N_GEN,
    seeds=(10, 11, 12, 13, 14),  # 5 seeds para a calibracao
)

ref_diversidade = construir_referencia_a_partir_da_trajetoria(traj_div)
ref_qualidade = construir_referencia_a_partir_da_trajetoria(traj_qual)

elapsed_ref = time.time() - t0_ref
print(f'Calibracao concluida em {elapsed_ref:.1f}s')
print(f'Makespan medio na calibracao: {resumo_calib["makespan_medio"]:.1f}')
print(f'  Trajetoria diversidade (10 primeiros): {np.round(traj_div[:10], 4)}')
print(f'  Trajetoria qualidade (10 primeiros):   {np.round(traj_qual[:10], 4)}')

>>> Calibrando referencias fuzzy...
Calibracao concluida em 16.5s
Makespan medio na calibracao: 1428.8
  Trajetoria diversidade (10 primeiros): [0.3325 0.3072 0.285  0.2629 0.2492 0.2279 0.1977 0.1707 0.1541 0.1341]
  Trajetoria qualidade (10 primeiros):   [0.     0.0465 0.0798 0.1211 0.1589 0.1778 0.1939 0.207  0.2259 0.2405]


## 5. Fábrica de Variantes (closures)

Cada variante é uma callable `(engine, seed) -> SolverResult`.

In [10]:
def fabricar_variante_fixo_default() -> Callable:
    """BRKGA com parametros de fabrica (20/70/10, bias 0.7)."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_default',
            n_gen=N_GEN,
            n_elites=N_ELITES_DEFAULT,
            n_offsprings=N_OFFSPRINGS_DEFAULT,
            n_mutants=N_MUTANTS_DEFAULT,
            bias=BIAS_DEFAULT,
        )
    return _run


def fabricar_variante_fixo_optuna(config: dict) -> Callable:
    """BRKGA com parametros encontrados pelo Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_optuna',
            n_gen=N_GEN,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
        )
    return _run


def fabricar_variante_fuzzy(
    config: dict,
    ref_div: Callable,
    ref_qual: Callable,
) -> Callable:
    """BRKGA adaptativo com controle fuzzy, partindo da config do Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fuzzy',
            n_gen=N_GEN,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
            ref_diversidade=ref_div,
            ref_qualidade=ref_qual,
        )
    return _run


# Monta o dicionario de variantes
variantes = {
    'fixo_default': fabricar_variante_fixo_default(),
    'fixo_optuna': fabricar_variante_fixo_optuna(melhor_config),
    'fuzzy': fabricar_variante_fuzzy(melhor_config, ref_diversidade, ref_qualidade),
}

print(f'Variantes disponiveis: {list(variantes.keys())}')

Variantes disponiveis: ['fixo_default', 'fixo_optuna', 'fuzzy']


## 6. Loop Comparativo -> 60 instâncias × 5 seeds × 3 variantes

Para cada instância:
- Carrega a engine
- Roda cada variante × seed
- salva tudo no csv

In [11]:
def amostrar_historico(history: list, n_gen: int, passo: int = 20) -> dict:
    """Extrai o makespan nos marcos gen=passo, 2*passo, ..., n_gen."""
    amostras = {}
    for gen in range(passo, n_gen + 1, passo):
        idx = min(gen - 1, len(history) - 1)
        amostras[f'mks_gen{gen}'] = float(history[idx])
    return amostras


# Schema fixo do CSV
CAMPOS_CSV = [
    'instancia', 'grupo', 'n_jobs', 'n_machines',
    'metodo', 'seed',
    'makespan', 'tardiness', 'cfo_count', 'tempo_s',
]
# Campos de convergencia (a cada 20 geracoes)
CAMPOS_CONVERGENCIA = [f'mks_gen{g}' for g in range(20, N_GEN + 1, 20)]
CAMPOS_CSV.extend(CAMPOS_CONVERGENCIA)

CSV_PATH = RESULTADOS_DIR / 'resultados_brkga_foco.csv'
print(f'CSV de saida: {CSV_PATH}')
print(f'Campos: {CAMPOS_CSV}')

CSV de saida: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_foco/resultados_brkga_foco.csv
Campos: ['instancia', 'grupo', 'n_jobs', 'n_machines', 'metodo', 'seed', 'makespan', 'tardiness', 'cfo_count', 'tempo_s', 'mks_gen20', 'mks_gen40', 'mks_gen60', 'mks_gen80', 'mks_gen100']


In [12]:
print('=' * 72)
print('INICIANDO LOOP COMPARATIVO')
print(f'{len(catalogo)} instancias x {len(variantes)} variantes x {N_SEEDS} seeds')
print(f'Total: {len(catalogo) * len(variantes) * N_SEEDS} execucoes')
print('=' * 72)

t0_total = time.time()

# Flag para escrever header apenas na primeira leva
csv_header_escrito = False

for idx_inst, (_, row) in enumerate(catalogo.iterrows()):
    inst_label = row['rotulo']
    inst_path = row['path']
    n_jobs = row['n_jobs']
    n_mach = row['n_machines']
    grupo = row['grupo']

    # Carrega engine uma vez por instancia
    nj, nm, pt, dd = FlowShopEngine.carregar_instancia_txt(inst_path)
    engine = FlowShopEngine(nj, nm, pt, dd)

    for metodo, fn_variante in variantes.items():
        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            # LOG de progresso detalhado
            print(
                f'[{idx_inst + 1}/{len(catalogo)}] '
                f'Inst: {row["arquivo"]} ({n_jobs}j x {n_mach}m, g{grupo}) '
                f'| Var: {metodo} '
                f'| Seed: {seed} ...',
                end=' '
            )

            try:
                t0_exec = time.time()
                result = fn_variante(engine, seed)
                elapsed = time.time() - t0_exec

                # Monta linha do CSV
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float(result.best_cost),
                    'tardiness': float(result.tardiness),
                    'cfo_count': int(result.engine_cfo_count),
                    'tempo_s': round(elapsed, 3),
                }
                # Amostras de convergencia
                amostras = amostrar_historico(result.history, N_GEN, 20)
                linha.update(amostras)

                print(f'OK  Makespan: {result.best_cost:.1f} ({elapsed:.2f}s)')

            except Exception as e:
                print(f'ERRO: {e}')
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float('nan'),
                    'tardiness': float('nan'),
                    'cfo_count': 0,
                    'tempo_s': float('nan'),
                }
                for c in CAMPOS_CONVERGENCIA:
                    linha[c] = float('nan')

            # Escreve no CSV incremental (abre/fecha a cada linha = seguro)
            escrever_header = not csv_header_escrito
            with open(CSV_PATH, 'w' if escrever_header else 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
                if escrever_header:
                    writer.writeheader()
                    csv_header_escrito = True
                writer.writerow(linha)

elapsed_total = time.time() - t0_total
print('=' * 72)
print(f'LOOP CONCLUIDO em {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)')
print(f'Resultados salvos em: {CSV_PATH}')

INICIANDO LOOP COMPARATIVO
60 instancias x 3 variantes x 5 seeds
Total: 900 execucoes
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_default | Seed: 100 ... OK  Makespan: 1008.0 (3.14s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_default | Seed: 101 ... OK  Makespan: 1035.0 (2.05s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_default | Seed: 102 ... OK  Makespan: 1047.0 (1.84s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_default | Seed: 103 ... OK  Makespan: 1006.0 (1.70s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_default | Seed: 104 ... OK  Makespan: 996.0 (1.73s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_optuna | Seed: 100 ... OK  Makespan: 987.0 (2.28s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_optuna | Seed: 101 ... OK  Makespan: 1008.0 (2.51s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_optuna | Seed: 102 ... OK  Makespan: 1047.0 (3.30s)
[1/60] Inst: I_2_16_2_1.txt (16j x 2m, g2) | Var: fixo_

## 7. Resumo Final

Carrega o CSV consolidado e exibe a tabela comparativa
agregada por instância × método.

In [13]:
df = pd.read_csv(CSV_PATH)
print(f'Total de linhas no CSV: {len(df)}')
print()

print('=' * 90)
print('TABELA COMPARATIVA  Makespan medio por Instancia x Metodo')
print('=' * 90)

# Agrupa por instancia + metodo
agg = df.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo'])['makespan'].agg(
    ['mean', 'std', 'min', 'max']
).reset_index()
agg.columns = ['instancia', 'n_jobs', 'n_mach', 'metodo', 'media', 'desvio', 'melhor', 'pior']
agg['media'] = agg['media'].round(1)
agg['desvio'] = agg['desvio'].round(1)

print(agg.to_string(index=False))
print()

# Resumo global
global_agg = df.groupby('metodo')['makespan'].agg(['mean', 'std', 'min']).reset_index()
global_agg.columns = ['metodo', 'makespan_medio', 'desvio_std', 'melhor_global']
global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['desvio_std'] = global_agg['desvio_std'].round(1)

print('=' * 60)
print('RESUMO GLOBAL (todas as 60 instancias, todas as seeds)')
print('=' * 60)
print(global_agg.to_string(index=False))
print()
print(f'Arquivo salvo em: {CSV_PATH}')

Total de linhas no CSV: 900

TABELA COMPARATIVA  Makespan medio por Instancia x Metodo
     instancia  n_jobs  n_mach       metodo  media  desvio  melhor   pior
I_2_16_2_1.txt      16       2 fixo_default 1018.4    21.5   996.0 1047.0
I_2_16_2_1.txt      16       2  fixo_optuna 1015.0    22.8   987.0 1047.0
I_2_16_2_1.txt      16       2        fuzzy 1023.2    34.8   983.0 1056.0
I_2_16_2_2.txt      16       2 fixo_default  637.8     1.8   637.0  641.0
I_2_16_2_2.txt      16       2  fixo_optuna  641.4     7.7   637.0  655.0
I_2_16_2_2.txt      16       2        fuzzy  656.4    17.7   637.0  681.0
I_2_16_2_3.txt      16       2 fixo_default  982.2     2.9   980.0  987.0
I_2_16_2_3.txt      16       2  fixo_optuna  987.0     9.6   982.0 1004.0
I_2_16_2_3.txt      16       2        fuzzy  986.2    10.2   980.0 1004.0
I_2_16_2_4.txt      16       2 fixo_default  859.2     8.7   848.0  869.0
I_2_16_2_4.txt      16       2  fixo_optuna  866.0    10.7   848.0  876.0
I_2_16_2_4.txt      16   